# ShotShield v1 (original)

The first version of this project, kept unchanged for comparison. It trains a CNN to tell **gunshots from dog barks** on UrbanSound8K.

Known issues, all fixed in `shotshield_v2_training.ipynb` (see the README):
- only 2 of the 10 classes are used, so it isn't a general gunshot detector
- short clips are padded with 0 dB, the loudest value, which lets the model learn clip length instead of the sound
- a random train/test split puts slices of the same recording on both sides
- accuracy is the only metric

In [ ]:
!pip install librosa soundfile scikit-learn tensorflow --quiet

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.utils import to_categorical
from IPython.display import Audio

In [ ]:
# Load UrbanSound8K metadata
metadata = pd.read_csv('/kaggle/input/urbansound8k/metadata/UrbanSound8K.csv')
metadata.head()

In [ ]:
# Example: Use gun_shot and dog_bark for binary classification
wanted_classes = ['gun_shot', 'dog_bark']

filtered_meta = metadata[metadata['class'].isin(wanted_classes)].reset_index(drop=True)
print(filtered_meta['class'].value_counts())

In [ ]:
# Example: Visualize one audio file
example_row = filtered_meta.iloc[0]
fold = example_row['fold']
file_name = example_row['slice_file_name']
file_path = f'/kaggle/input/urbansound8k/audio/fold{fold}/{file_name}'

# Load waveform
signal, sr = librosa.load(file_path)

# Display waveform
plt.figure(figsize=(10, 4))
librosa.display.waveshow(signal, sr=sr)
plt.title('Waveform')
plt.show()

# Display spectrogram
spectrogram = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
spectrogram_db = librosa.power_to_db(spectrogram, ref=np.max)

plt.figure(figsize=(10, 4))
librosa.display.specshow(spectrogram_db, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.show()

# Play audio
Audio(file_path)

In [ ]:
def extract_spectrogram(file_path, max_pad_len=174):
    signal, sr = librosa.load(file_path, res_type='kaiser_fast')
    melspec = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    log_melspec = librosa.power_to_db(melspec, ref=np.max)
    
    # Padding/truncate
    if log_melspec.shape[1] < max_pad_len:
        pad_width = max_pad_len - log_melspec.shape[1]
        log_melspec = np.pad(log_melspec, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        log_melspec = log_melspec[:, :max_pad_len]
    return log_melspec

In [ ]:
X = []
y = []

for i, row in filtered_meta.iterrows():
    fold = row['fold']
    file_name = row['slice_file_name']
    file_path = f'/kaggle/input/urbansound8k/audio/fold{fold}/{file_name}'
    try:
        spectrogram = extract_spectrogram(file_path)
        X.append(spectrogram)
        y.append(row['class'])
    except Exception as e:
        print(f"Error with {file_name}: {e}")

X = np.array(X)
y = np.array(y)

# Add channel dimension for CNN
X = X[..., np.newaxis]

# Encode labels
le = LabelEncoder()
y_encoded = to_categorical(le.fit_transform(y))

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 174, 1)))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.3))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.3))

model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
# Pick an example
example_row = filtered_meta.iloc[1]
fold = example_row['fold']
file_name = example_row['slice_file_name']
file_path = f'/kaggle/input/urbansound8k/audio/fold{fold}/{file_name}'

spectrogram = extract_spectrogram(file_path).reshape(1, 128, 174, 1)
prediction = model.predict(spectrogram)
predicted_label = le.inverse_transform([np.argmax(prediction)])

print(f"Actual: {example_row['class']} | Predicted: {predicted_label[0]}")

# Play it
Audio(file_path)